In [ ]:
# ========================================== 
# UNIVERSAL INVENTORY MANAGEMENT SYSTEM FOR SMEs
# ==========================================

import pandas as pd
from datetime import datetime
from tabulate import tabulate
import os

# --- BASIC USER AUTHENTICATION ---
def login():
    """Simple authentication to secure system access."""
    password = "admin123"  # You can change this
    attempts = 3
    while attempts > 0:
        user_input = input("Enter system password: ")
        if user_input == password:
            print("🔐 Access granted. Welcome to the Inventory Management System!\n")
            return True
        else:
            attempts -= 1
            print(f"❌ Incorrect password. {attempts} attempt(s) left.")
    print("🚫 Access denied. Exiting program...")
    exit()



# --- AUTO-LOAD EXISTING DATA ---
def load_inventory():
    if os.path.exists("inventory_autosave.csv"):
        print("✅ Existing inventory data loaded successfully.")
        return pd.read_csv("inventory_autosave.csv")
    else:
        print("⚠️ No existing inventory file found. Starting new inventory.")
        return pd.DataFrame(columns=[
            'Product ID', 'Product Name', 'Category', 'Quantity Ordered',
            'Quantity in Stock', 'Quantity Sold', 'Cost Price', 'Unit Price',
            'Supplier', 'Date of Entry'
        ])

# --- AUTO-SAVE FUNCTION ---
def autosave_inventory(inventory_df):
    inventory_df.to_csv("inventory_autosave.csv", index=False)
    print("💾 Progress saved automatically.")

# --- SAFE MONEY INPUT CLEANER ---
def clean_money_input(prompt):
    """Cleans and converts any money input to float safely."""
    while True:
        user_input = input(prompt)
        try:
            clean_value = (
                user_input.replace(",", "")
                .replace("₦", "")
                .replace("$", "")
                .replace("€", "")
                .replace("£", "")
                .strip()
            )
            return float(clean_value)
        except ValueError:
            print("❌ Invalid input. Please enter a valid number (e.g., 50000 or 50000.75).")



# --- 1. ADD NEW PRODUCT ---
def add_product(inventory_df):
    print("\n--- ADD NEW PRODUCT ---")
    new_id = "P" + str(len(inventory_df) + 1).zfill(3)
    print(f"Auto-generated Product ID: {new_id}")

    product_name = (input("Enter Product Name: "))
    category = (input("Enter Product Category: "))
    qty_ordered = int(input("Enter Quantity Ordered: "))
    qty_stock = int(input("Enter Quantity in Stock: "))
    qty_sold = int(input("Enter Quantity Sold: "))
    cost_price = clean_money_input("Enter Cost Price: ")
    unit_price = clean_money_input("Enter Unit Price: ")
    supplier = (input("Enter Supplier Name: "))
    date_entry = input("Enter Date of Entry (YYYY-MM-DD): ")

    new_row = {
        'Product ID': new_id,
        'Product Name': product_name,
        'Category': category,
        'Quantity Ordered': qty_ordered,
        'Quantity in Stock': qty_stock,
        'Quantity Sold': qty_sold,
        'Cost Price': cost_price,
        'Unit Price': unit_price,
        'Supplier': supplier,
        'Date of Entry': date_entry
    }

    inventory_df.loc[len(inventory_df)] = new_row
    autosave_inventory(inventory_df)
    print("\n✅ Product added successfully and saved!\n")
    print("💾 Changes saved automatically to 'inventory_autosave.csv'")
    return inventory_df
    
# --- 2. VIEW PRODUCTS ---
def view_products(inventory_df):
    print("\n--- INVENTORY RECORDS ---")
    if inventory_df.empty:
        print("No records available.")
    else:
        print(inventory_df)

def update_product(inventory_df):
    print("\n--- UPDATE PRODUCT(S) ---")
    
    # Allow updating multiple products at once
    product_ids = input("Enter Product ID(s) to update (separate multiple IDs with commas): ").split(',')
    product_ids = [pid.strip() for pid in product_ids]

    for product_id in product_ids:
        if product_id not in inventory_df['Product ID'].values:
            print(f"❌ Product ID '{product_id}' not found. Skipping...\n")
            continue

        print(f"\nEditing Product ID: {product_id}")
        print(inventory_df[inventory_df['Product ID'] == product_id])

        # Allow updating multiple columns for this product
        for column in inventory_df.columns:
            if column == 'Product ID':  # skip ID updates
                continue
            current_value = inventory_df.loc[inventory_df['Product ID'] == product_id, column].values[0]
            new_value = input(f"Enter new value for '{column}' (current: {current_value}) or press Enter to skip: ")

            if new_value.strip() != "":
                # Auto-convert numbers where possible
                try:
                    if '.' in new_value:
                        new_value = float(new_value)
                    elif new_value.isdigit():
                        new_value = int(new_value)
                except ValueError:
                    pass
                inventory_df.loc[inventory_df['Product ID'] == product_id, column] = new_value

        print(f"\n✅ Updates completed for Product ID '{product_id}'.\n")

    print("All selected product updates are complete.\n")
    print("💾 Changes saved automatically to 'inventory_autosave.csv'")
    return inventory_df

# --- REAL-TIME STOCK TRACKING ---
def update_stock_after_sale(inventory_df):
    """Automatically updates remaining stock when Quantity Sold changes."""
    if 'Quantity Sold' in inventory_df.columns and 'Quantity in Stock' in inventory_df.columns:
        inventory_df['Quantity in Stock'] = inventory_df['Quantity Ordered'] - inventory_df['Quantity Sold']
        autosave_inventory(inventory_df)
        print("🔄 Stock levels updated automatically based on sales data.")
    else:
        print("⚠️ Columns for Quantity Ordered/Sold/Stock missing. Skipping real-time update.")



# --- 4. DELETE PRODUCT ---
def delete_product(inventory_df):
    print("\n--- DELETE PRODUCT ---")
    product_id = input("Enter Product ID to delete: ")
    if product_id in inventory_df['Product ID'].values:
        inventory_df.drop(inventory_df[inventory_df['Product ID'] == product_id].index, inplace=True)
        autosave_inventory(inventory_df)
        print("\n✅ Product deleted successfully!\n")
        print("💾 Changes saved automatically to 'inventory_autosave.csv'")
    else:
        print("Product ID not found.")
    return inventory_df

# --- 5. ADD NEW COLUMN ---
def add_new_column(inventory_df):
    print("\n--- ADD NEW COLUMN ---")
    new_col = input("Enter new column name: ")
    default_value = input("Enter default value for existing rows (or press Enter to leave blank): ")
    inventory_df[new_col] = default_value if default_value != "" else None
    autosave_inventory(inventory_df)
    print(f"\n✅ New column '{new_col}' added successfully and saved!\n")
    print("💾 Changes saved automatically to 'inventory_autosave.csv'")
    return inventory_df

def analytics_summary(inventory_df):
    print("\n=== INVENTORY ANALYTICS SUMMARY ===")
    if inventory_df.empty:
        print("No data available for analysis.")
        return inventory_df

    # --- PROFIT AND STOCK VALUE COMPUTATION ---
    inventory_df['Profit'] = (inventory_df['Unit Price'] - inventory_df['Cost Price']) * inventory_df['Quantity Sold']
    inventory_df['Stock Value'] = inventory_df['Quantity in Stock'] * inventory_df['Cost Price']
    total_profit = inventory_df['Profit'].sum()
    total_stock_value = inventory_df['Stock Value'].sum()

    # --- LOW STOCK ALERTS ---
    low_stock = inventory_df[inventory_df['Quantity in Stock'] < 10]
    avg_price = inventory_df['Unit Price'].mean()
    category_stock = inventory_df.groupby('Category')['Quantity in Stock'].sum().idxmax()

    print("\n--- PROFIT BY PRODUCT ---")
    print(inventory_df[['Product ID', 'Product Name', 'Profit']])

    print(f"\n💰 Total Profit: ${total_profit:,.2f}")
    print(f"📦 Total Stock Value: ${total_stock_value:,.2f}")
    print(f"💲 Average Unit Price: ${avg_price:,.2f}")
    print(f"🏷️ Category with Most Stock: {category_stock}")

    # --- RESTOCK RECOMMENDATION ---
    if not low_stock.empty:
        low_stock['Reorder Suggestion'] = 20 - low_stock['Quantity in Stock']
        low_stock['Reorder Suggestion'] = low_stock['Reorder Suggestion'].apply(lambda x: x if x > 0 else 0)
        print("\n--- LOW STOCK & REORDER SUGGESTIONS ---")
        print(low_stock[['Product ID', 'Product Name', 'Quantity in Stock', 'Reorder Suggestion']])
    else:
        print("\nAll products have sufficient stock levels.")

    # --- SUPPLIER ANALYTICS ---
    if 'Supplier' in inventory_df.columns:
        top_supplier = inventory_df['Supplier'].mode()[0]
        print(f"\n📊 Supplier with Most Products: {top_supplier}")
        supplier_counts = inventory_df['Supplier'].value_counts()
        print("\n--- PRODUCTS BY SUPPLIER ---")
        print(supplier_counts)

    return inventory_df

# --- 7. SAVE DATASET MANUALLY ---
def save_dataset(inventory_df):
    print("\n--- SAVE DATASET ---")
    file_name = input("Enter filename to save (e.g., inventory_data.csv): ")
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    file_name = file_name.replace(".csv", f"_{timestamp}.csv")
    inventory_df.to_csv(file_name, index=False)
    print(f"\n✅ Dataset saved successfully as '{file_name}' in your working directory!\n")

# --- SESSION PERFORMANCE LOG ---
def log_session_summary(inventory_df):
    """Creates a session log for validation and experimental tracking."""
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    log_data = {
        'Timestamp': timestamp,
        'Total_Products': len(inventory_df),
        'Total_Profit': inventory_df['Profit'].sum() if 'Profit' in inventory_df.columns else 0,
        'Low_Stock_Items': len(inventory_df[inventory_df['Quantity in Stock'] < 10])}
    log_df = pd.DataFrame([log_data])
    if os.path.exists("session_log.csv"):
        log_df.to_csv("session_log.csv", mode='a', header=False, index=False)
    else:
        log_df.to_csv("session_log.csv", index=False)
    print("🧾 Session summary logged for validation.")

# --- MAIN MENU ---
def main_menu(inventory_df):
    while True:
        print("\n==== INVENTORY MANAGEMENT SYSTEM ====")
        print("1. Add New Product")
        print("2. View All Products")
        print("3. Update Product Details")
        print("4. Delete Product")
        print("5. Add New Column")
        print("6. View Analytics Summary")
        print("7. Save Dataset to CSV")
        print("8. Exit")

        choice = input("Enter your choice (1–8): ")

        if choice == '1':
            inventory_df = add_product(inventory_df)
        elif choice == '2':
            view_products(inventory_df)
        elif choice == '3':
            inventory_df = update_product(inventory_df)
        elif choice == '4':
            inventory_df = delete_product(inventory_df)
        elif choice == '5':
            inventory_df = add_new_column(inventory_df)
        elif choice == '6':
            inventory_df = analytics_summary(inventory_df)
        elif choice == '7':
            save_dataset(inventory_df)
        elif choice == '8':
            print("\nExiting program. Goodbye!\n")
            break
        else:
            print("\nInvalid choice. Please try again.")
    return inventory_df

# --- RUN PROGRAM ---
login()
inventory_df = load_inventory()
main_menu(inventory_df)

Enter system password:  admin123


🔐 Access granted. Welcome to the Inventory Management System!

✅ Existing inventory data loaded successfully.

==== INVENTORY MANAGEMENT SYSTEM ====
1. Add New Product
2. View All Products
3. Update Product Details
4. Delete Product
5. Add New Column
6. View Analytics Summary
7. Save Dataset to CSV
8. Exit
